# 04 - Agent Build & Test

**Pre-req**: Notebooks 00–03 complete.

In [1]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Confirm key env vars loaded
assert os.getenv('GROQ_API_KEY'), 'GROQ_API_KEY missing from .env'
print(f'GROQ key loaded: {os.getenv("GROQ_API_KEY")[:8]}...')

# agent.py and tools.py are in project root
sys.path.insert(0, str(Path('..').resolve()))

from agent import create_agent, run_agent, run_agent_stream
print('Imports OK')

GROQ key loaded: gsk_vKiL...
Imports OK


## 1. Build agent

In [2]:
agent = create_agent()
print('Agent ready')
print("Tools: ['weaviate_search', 'neo4j_graph_search']")

Agent ready
Tools: ['weaviate_search', 'neo4j_graph_search']


C:\Users\abhir\Downloads\graph-rag-local\agent.py:44: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(


## 2. Single turn

In [3]:
response = run_agent(
    agent,
    query='What are the main supply chain risks for electronics manufacturers?',
    thread_id='test-thread-001',
)
print(response)

According to [paper], the main supply chain risks for electronics manufacturers include disruptions in transportation, supplier insolvency, and hold-up risks. Graph: Manufacturing --[PART_OF]--> Distribution, and Manufacturing --[PART_OF]--> supply chains, indicate that manufacturing is a critical component of the supply chain. Additionally, Graph: Supply Chain --[SHIPS_TO]--> Shipments, and Supply Chain --[MANAGES]--> DC1, highlight the importance of logistics and shipment management in the supply chain. Furthermore, Graph: Risks --[AFFECTS]--> Businesses, suggests that risks in the supply chain can have a significant impact on businesses. 

To mitigate these risks, electronics manufacturers can consider implementing strategies such as supplier diversification, risk assessment, and contingency planning. According to [paper], it is also important to examine the safety and lead times of shipment schedules, as well as label and examine potential secondary carriers or routes. 

In summary

## 3. Multi-turn (memory via PostgresSaver)

In [4]:
THREAD = 'test-multiturn-001'

r1 = run_agent(agent, 'What is just-in-time inventory?', THREAD)
print('Turn 1:', r1[:300])

r2 = run_agent(agent, 'How does it affect supplier relationships?', THREAD)
print('\nTurn 2 (uses context from turn 1):')
print(r2[:300])

Turn 1: According to [supply_chain_management_tutorial.pdf], just-in-time inventory is a management strategy that aims to maintain minimal inventory levels by producing and receiving inventory just in time to meet customer demand. This approach helps companies minimize waste, reduce costs, and improve effic

Turn 2 (uses context from turn 1):
According to [supply_chain_management_tutorial.pdf], just-in-time inventory management can affect supplier relationships by exposing the heart of these relationships, which is inventory movement and storage. More than half of supply chain relationships rely on the purchase, transfer, or management o


## 4. Streaming

In [5]:
print('Streaming:')
print('-' * 60)
full = ''
for chunk in run_agent_stream(agent, 'Which suppliers are at risk from port congestion?', 'test-stream-001'):
    print(chunk, end='', flush=True)
    full += chunk
print(f'\nTotal chars: {len(full)}')

Streaming:
------------------------------------------------------------
According to [1], port congestion can trigger widespread instability across the entire supply chain system. Graph: Transportation --[SHIPS_TO]--> Customer, and Transportation --[SHIPS_TO]--> Cafe 2, indicate that suppliers who ship to these destinations are at risk from port congestion. 

As [2] states, modern supply chain management involves managing intricate interdependencies across multi-level networks. Therefore, it is crucial to identify and mitigate potential risks, such as port congestion, to minimize downtime and ensure operational resilience. 

[3] suggests that graph algorithms can be used to optimize routes and identify critical nodes, which can help suppliers reroute shipments during port closures. Additionally, machine learning models can be used to predict demand and manage disruptions, enhancing resilience and adaptability. 

As [4] notes, it is essential to acknowledge the risks related to the loca

## 5. Verify checkpoints

In [6]:
import os, psycopg
from urllib.parse import quote

DB_URI = (
    f'postgresql://{quote(os.getenv("PG_USER"), safe="")}:{quote(os.getenv("PG_PASSWORD"), safe="")}'
    f'@{os.getenv("PG_HOST","localhost")}:{os.getenv("PG_PORT","5432")}/{os.getenv("PG_DATABASE","postgres")}'
)

with psycopg.connect(DB_URI) as conn:
    rows = conn.execute(
        'SELECT thread_id, COUNT(*) AS c FROM checkpoints GROUP BY thread_id ORDER BY c DESC'
    ).fetchall()
    print('Checkpoints per thread:')
    for thread_id, count in rows:
        print(f'  {thread_id}: {count}')

Checkpoints per thread:
  test-multiturn-001: 10
  test-thread-001: 5
  test-stream-001: 5
